# T2I-RL Quick Start Guide

This notebook demonstrates how to use the T2I-RL framework for training text-to-image models with reinforcement learning using VLM-based rewards.

## Overview

1. **Setup & Installation**
2. **Load Janus-Pro Generator**
3. **Generate Images with Log Probabilities**
4. **Compute Rewards using VLM**
5. **Run GRPO Training Step**
6. **Evaluate on Benchmarks**

## 1. Setup & Installation

First, ensure you have all dependencies installed:

In [ ]:
# Install dependencies (run once)
# !pip install -r ../requirements.txt
# !pip install git+https://github.com/deepseek-ai/Janus.git

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from PIL import Image
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load Janus-Pro Generator

We use Janus-Pro-1B as our base text-to-image generator. It's an autoregressive model that generates images as discrete visual tokens.

In [ ]:
from src.models.generators import JanusProGenerator

# Initialize the generator
generator = JanusProGenerator(
    model_name="deepseek-ai/Janus-Pro-1B",
    device=device,
    use_lora=False,  # Set True for LoRA fine-tuning
    torch_dtype=torch.bfloat16  # Use bfloat16 for efficiency
)

print(f"Model loaded: {generator.model_name}")
print(f"Image size: {generator.image_size}x{generator.image_size}")
print(f"Visual tokens: {generator.num_image_tokens}")

## 3. Generate Images

### 3.1 Basic Generation

In [ ]:
# Simple image generation
prompts = [
    "A red apple on a blue plate",
    "A cat sitting next to a dog in a garden",
    "Three yellow bananas on a wooden table"
]

images = generator.generate(
    prompts=prompts,
    cfg_weight=5.0,  # Classifier-free guidance weight
    temperature=1.0,
    num_inference_steps=1  # Autoregressive, so 1 step
)

# Display generated images
fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 5))
for ax, img, prompt in zip(axes, images, prompts):
    ax.imshow(img)
    ax.set_title(prompt[:30] + '...' if len(prompt) > 30 else prompt)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 3.2 Generation with Log Probabilities (for RL Training)

For GRPO training, we need to track the log probabilities of generated tokens:

In [ ]:
# Generate with log probabilities for RL
prompts = ["A blue car parked in front of a red house"]

images, log_probs, generated_ids = generator.generate_with_logprobs(
    prompts=prompts,
    cfg_weight=5.0,
    temperature=1.0
)

print(f"Generated image shape: {images[0].size}")
print(f"Log probs shape: {log_probs.shape}")
print(f"Generated token IDs shape: {generated_ids.shape}")
print(f"Mean log probability: {log_probs.mean().item():.4f}")

plt.figure(figsize=(6, 6))
plt.imshow(images[0])
plt.title(prompts[0])
plt.axis('off')
plt.show()

## 4. Compute Rewards using VLM

We use VLM-based rewards to evaluate how well the generated image matches the prompt. This provides understanding-based feedback.

In [ ]:
from src.models.reward_models import CLIPRewardModel, VLMRewardModel

# Option 1: CLIP-based reward (fast, no API needed)
clip_reward = CLIPRewardModel(
    model_name="openai/clip-vit-large-patch14",
    device=device
)

# Compute CLIP similarity reward
clip_scores = clip_reward.compute_reward(images, prompts)
print(f"CLIP Reward: {clip_scores}")

In [ ]:
# Option 2: VLM-based reward (requires API key)
# Uncomment and set your API key to use

# import os
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

# vlm_reward = VLMRewardModel(
#     model_name="gpt-4-vision-preview",
#     provider="openai"
# )

# vlm_scores = vlm_reward.compute_reward(images, prompts)
# print(f"VLM Reward: {vlm_scores}")

### 4.1 Understanding the VLM Reward

The VLM reward evaluates multiple aspects:
- **Object Presence**: Are all mentioned objects in the image?
- **Attribute Binding**: Do objects have the correct attributes (colors, sizes)?
- **Spatial Relations**: Are objects positioned correctly relative to each other?
- **Overall Quality**: Image quality and prompt faithfulness

In [ ]:
# Example: Detailed VLM evaluation (conceptual)
evaluation_prompt = """
Evaluate this image against the prompt: "{prompt}"

Score each aspect from 0-10:
1. Object Presence: Are all mentioned objects present?
2. Attribute Accuracy: Do objects have correct colors/sizes?
3. Spatial Relations: Are objects positioned correctly?
4. Overall Quality: Image quality and prompt faithfulness

Return scores as JSON: {{"presence": X, "attributes": X, "spatial": X, "quality": X}}
"""

print("VLM Evaluation Prompt Template:")
print(evaluation_prompt.format(prompt=prompts[0]))

## 5. Run GRPO Training Step

GRPO (Group Relative Policy Optimization) generates multiple samples per prompt and uses relative rewards within the group.

In [ ]:
from src.training.grpo_trainer import GRPOTrainer
from omegaconf import OmegaConf

# Load configuration
config = OmegaConf.create({
    'training': {
        'learning_rate': 1e-5,
        'batch_size': 2,
        'gradient_accumulation_steps': 4,
        'max_grad_norm': 1.0,
        'grpo': {
            'group_size': 4,  # Generate 4 samples per prompt
            'kl_coef': 0.01,
            'clip_range': 0.2,
            'normalize_rewards': True
        }
    },
    'model': {
        'generator': {
            'name': 'deepseek-ai/Janus-Pro-1B',
            'use_lora': True,
            'lora_rank': 16
        }
    }
})

print("GRPO Configuration:")
print(OmegaConf.to_yaml(config))

In [ ]:
# Initialize GRPO trainer (demonstration)
# Note: Full training requires GPU and significant compute

# trainer = GRPOTrainer(
#     config=config,
#     generator=generator,
#     reward_model=clip_reward,
#     device=device
# )

# # Single training step
# prompts_batch = [
#     "A red sphere on a blue cube",
#     "Two cats playing with a ball"
# ]
# 
# loss, metrics = trainer.training_step(prompts_batch)
# print(f"Training Loss: {loss:.4f}")
# print(f"Metrics: {metrics}")

### 5.1 GRPO Algorithm Explanation

```
For each prompt p:
  1. Generate K images: {x_1, ..., x_K} ~ π_θ(x|p)
  2. Compute rewards: {r_1, ..., r_K}
  3. Normalize rewards within group: r̂_i = (r_i - mean(r)) / std(r)
  4. Compute GRPO loss:
     L = -E[r̂_i * log π_θ(x_i|p)] + β * KL(π_θ || π_ref)
```

## 6. Evaluate on Benchmarks

We support evaluation on standard T2I benchmarks:

In [ ]:
from src.evaluation.benchmarks import T2ICompBenchEvaluator

# T2I-CompBench evaluation categories
categories = [
    "color_binding",      # "a red car and a blue house"
    "shape_binding",      # "a circular table and a square chair"
    "texture_binding",    # "a wooden desk and a metal lamp"
    "spatial_relations",  # "a cat on top of a box"
    "non_spatial",        # "a dog playing with a cat"
    "complex_scenes"      # Multiple objects with attributes
]

print("T2I-CompBench Evaluation Categories:")
for cat in categories:
    print(f"  - {cat}")

In [ ]:
# Example evaluation prompts from T2I-CompBench
sample_prompts = {
    "color_binding": [
        "a red apple and a green pear",
        "a blue car and a yellow taxi",
        "a white cat and a black dog"
    ],
    "spatial_relations": [
        "a cat sitting on a chair",
        "a ball under the table",
        "a bird flying above the tree"
    ],
    "complex_scenes": [
        "a red apple on a blue plate next to a green cup",
        "two cats playing with a ball in front of a fireplace",
        "a wooden table with a metal lamp and a glass vase"
    ]
}

print("Sample Evaluation Prompts:")
for category, prompts in sample_prompts.items():
    print(f"\n{category}:")
    for p in prompts:
        print(f"  - {p}")

## 7. Full Training Pipeline

To run the full training pipeline, use the command line:

In [ ]:
# Training command
training_cmd = """
python scripts/train.py \\
    --config configs/default.yaml \\
    training.num_epochs=10 \\
    training.batch_size=4 \\
    model.generator.use_lora=true \\
    reward.type=vlm \\
    reward.vlm.provider=openai
"""

print("Training Command:")
print(training_cmd)

In [ ]:
# Evaluation command
eval_cmd = """
python scripts/evaluate.py \\
    --checkpoint outputs/checkpoints/best_model.pt \\
    --benchmark t2i_compbench \\
    --output_dir outputs/evaluation
"""

print("Evaluation Command:")
print(eval_cmd)

## 8. Summary

This notebook demonstrated:

1. **Generator Setup**: Loading Janus-Pro-1B for text-to-image generation
2. **Image Generation**: Basic generation and generation with log probabilities
3. **Reward Computation**: CLIP-based and VLM-based rewards
4. **GRPO Training**: Understanding the training algorithm
5. **Benchmark Evaluation**: T2I-CompBench categories and metrics

### Key Concepts

- **GRPO**: Generates multiple samples per prompt, uses relative rewards within the group
- **VLM Rewards**: Evaluate semantic understanding (object presence, attributes, spatial relations)
- **LoRA**: Efficient fine-tuning by training low-rank adaptation matrices

### Next Steps

1. Prepare your training prompts in `data/prompts/train_prompts.json`
2. Set up your API keys for VLM rewards (OpenAI/Anthropic)
3. Run training with `python scripts/train.py`
4. Evaluate on benchmarks with `python scripts/evaluate.py`

In [ ]:
print("Happy training! 🚀")